In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
'''
Loading Gensim and nltk libraries
'''
# pip install gensim
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(400)

In [3]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
df = pd.read_csv("/content/2022_Sep3_17k_hydrated_tweets.csv")

In [5]:
df.drop(columns=['id_str',"in_reply_to_screen_name","is_retweet"], inplace=True)

In [6]:
# Function to Clean the Tweet.

import re
def clean_tweet(tweet):
    return ' '.join(re.sub('(\\\\n)|(b\"[^0-9A-Za-z A-Za-z0-9 \t]+)|(b\'[^0-9A-Za-z]+)|(b\"[A-Za-z0-9]+)|(b\'[A-Za-z0-9]+)|(b\'#[A-Za-z0-9]+)|(b\'@[A-Za-z0-9]+)|(\\\\x[A-Za-z0-9]+)|(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|([RT])', ' ', str(tweet).lower()).split())


In [7]:
# Call function to get Clean tweets
df["CleanTweet"] = df['text'].apply(lambda x : clean_tweet(x))
df.tail()

,favorite_count,source,text,created_at,retweet_count,CleanTweet
15666,12,Twitter Web App,b'UK to begin new COVID-19 vaccination campaig...,Sun Sep 04 02:00:01 +0000 2022,4,to begin new covid 19 vaccination campaign
15667,0,WordPress.com,"b'(URGENT) S. Korea reports 72,144 new COVID-1...",Sun Sep 04 02:00:57 +0000 2022,0,urgent s korea reports 72 144 new covid 19 cases
15668,1,Twitter for Android,b'@ethanjweiss Thanks for your honesty in post...,Sun Sep 04 02:03:29 +0000 2022,0,ethanjweiss thanks for your honesty in posting...
15669,0,WordPress.com,b'Brockton to spray city following West Nile V...,Sun Sep 04 02:03:25 +0000 2022,0,to spray city following west nile virus discovery
15670,0,Twitter for iPhone,b'Antiviral Therapy | COVID-19 Treatment Guide...,Sun Sep 04 02:02:49 +0000 2022,0,therapy covid 19 treatment guidelines


In [8]:
df.drop(df.index[df.CleanTweet.eq("")], inplace=True)

In [ ]:
df.head()

,favorite_count,source,text,created_at,retweet_count,CleanTweet
0,1,Twitter for Android,"b""Don't believe it. https://t.co/VFfhPArn1a""",Sat Sep 03 04:09:04 +0000 2022,0,t believe it
1,0,cmssocialservice,b'Surat records 11 new Covid-19 cases https://...,Sat Sep 03 04:06:03 +0000 2022,0,records 11 new covid 19 cases
2,38,TweetDeck,b'Hennepin County will no longer require COVID...,Sat Sep 03 04:09:00 +0000 2022,6,county will no longer require covid 19 vaccine...
3,2,Twitter Web App,"b'As of Sept 1, the capital has given over 63....",Sat Sep 03 04:08:52 +0000 2022,0,of sept 1 the capital has given over 63 2 mill...
4,1,Twitter for Android,b'@KushThrough @BlazedRTs @rtsmallstreams @Rts...,Sat Sep 03 04:15:46 +0000 2022,3,kushthrough ww nowplaying x lilroyce her ev


In [9]:
stemmer = SnowballStemmer(language='english')
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

# Tokenize and lemmatize
def preprocess(text):
    result=[]
    for token in gensim.utils.simple_preprocess(text) :
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
            
    return result

In [10]:
complete_text = ' '.join(df["CleanTweet"])

In [11]:
data = df.CleanTweet.values.tolist()

In [ ]:
pprint(data[:2])

['t believe it', 'records 11 new covid 19 cases']


In [ ]:
complete_text


't believe it records 11 new covid 19 cases county will no longer require covid 19 vaccines for its employees of sept 1 the capital has given over 63 2 million doses of covid19 vaccines to over 23 6 million people kushthrough ww nowplaying x lilroyce her ev can t be good for you themixmedic nowplaying x lilroyce her evil spine feat makrazy total of 174 new cases of covid 19 were reported during the last 24 hours across the state odisha odishanews appointed by the israeli moh to investigate covid 19 vaccine side effects warned the ministry it could b covid 19 closed our campuses down the idea that every student has a computer a word processing program sta thetruthsucks12 why get poked in 2021 when covid 19 has a 98 survival rate got a bridge in brooklyn for sale yo federal government is fucking trash dubai reins in hospitality as covid 19 cases rise is hilarious you can make this up the camp counselor who tried to overturn a decisive democratic election pminmangaluru pm modi mentioned t

In [12]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [13]:
processed_docs = preprocess(complete_text)

In [14]:
'''
Create a dictionary from 'processed_docs' containing the number of times a word appears 
in the training set using gensim.corpora.Dictionary and call it 'dictionary'
'''
dictionary = gensim.corpora.Dictionary([processed_docs])

In [15]:
'''
Checking dictionary created
'''
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 aaaaaaaaa
1 aajtak
2 aapi
3 aapl
4 aarogya
5 aaron
6 aarondodd
7 aaronla
8 aaronparna
9 aarospeir
10 aayeff


In [ ]:
#dictionary.filter_extremes(no_below=20, no_above=0.1, keep_n= 1000000)
#dictionary.filter_extremes(no_above=0.70)

In [ ]:
dictionary

In [16]:
#bow_corpus = dictionary.doc2bow(processed_docs)

bow_corpus = [dictionary.doc2bow(processed_docs),]
#bow_corpus = [dictionary.doc2bow(text) for text in processed_docs]
print(bow_corpus[:1])

[[(0, 1), (1, 8), (2, 1), (3, 1), (4, 1), (5, 1), (6, 2), (7, 1), (8, 3), (9, 1), (10, 1), (11, 2), (12, 1), (13, 11), (14, 5), (15, 2), (16, 1), (17, 1), (18, 1), (19, 1), (20, 2), (21, 1), (22, 2), (23, 1), (24, 1), (25, 1), (26, 3), (27, 2), (28, 1), (29, 1), (30, 1), (31, 11), (32, 30), (33, 15), (34, 1), (35, 1), (36, 1), (37, 1), (38, 5), (39, 1), (40, 1), (41, 5), (42, 14), (43, 1), (44, 1), (45, 5), (46, 3), (47, 2), (48, 1), (49, 31), (50, 1), (51, 2), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 1), (59, 10), (60, 1), (61, 1), (62, 7), (63, 2), (64, 3), (65, 1), (66, 1), (67, 13), (68, 1), (69, 1), (70, 1), (71, 18), (72, 1), (73, 27), (74, 1), (75, 6), (76, 1), (77, 1), (78, 1), (79, 5), (80, 1), (81, 41), (82, 1), (83, 149), (84, 1), (85, 1), (86, 2), (87, 7), (88, 1), (89, 19), (90, 1), (91, 1), (92, 2), (93, 1), (94, 12), (95, 7), (96, 2), (97, 7), (98, 1), (99, 1), (100, 5), (101, 1), (102, 1), (103, 1), (104, 1), (105, 1), (106, 5), (107, 7), (108, 1), (10

In [17]:
#LDA
lda_model = gensim.models.ldamodel.LdaModel(corpus=bow_corpus, 
                                            num_topics = 20, 
                                            id2word = dictionary,
                                            random_state=100,
                                            update_every=1,
                                            chunksize=100,
                                            alpha='auto',
                                            per_word_topics=True,
                                            passes = 50)

In [18]:
for idx, topic in lda_model.print_topics(-1):
    print("Topic: {} \nWords: {}".format(idx, topic ))
    print("\n")

Topic: 0 
Words: 0.000*"covid" + 0.000*"vaccin" + 0.000*"case" + 0.000*"peopl" + 0.000*"death" + 0.000*"coronavirus" + 0.000*"report" + 0.000*"pandem" + 0.000*"time" + 0.000*"booster"


Topic: 1 
Words: 0.071*"covid" + 0.014*"vaccin" + 0.011*"case" + 0.010*"coronavirus" + 0.008*"peopl" + 0.007*"death" + 0.007*"report" + 0.006*"pandem" + 0.006*"booster" + 0.005*"trump"


Topic: 2 
Words: 0.000*"covid" + 0.000*"vaccin" + 0.000*"case" + 0.000*"coronavirus" + 0.000*"peopl" + 0.000*"death" + 0.000*"booster" + 0.000*"pandem" + 0.000*"report" + 0.000*"time"


Topic: 3 
Words: 0.000*"covid" + 0.000*"vaccin" + 0.000*"case" + 0.000*"coronavirus" + 0.000*"booster" + 0.000*"death" + 0.000*"peopl" + 0.000*"pandem" + 0.000*"report" + 0.000*"time"


Topic: 4 
Words: 0.000*"covid" + 0.000*"case" + 0.000*"peopl" + 0.000*"coronavirus" + 0.000*"vaccin" + 0.000*"booster" + 0.000*"death" + 0.000*"report" + 0.000*"time" + 0.000*"trump"


Topic: 5 
Words: 0.000*"covid" + 0.000*"vaccin" + 0.000*"case" + 0.000

In [19]:
import re
import numpy as np
import pandas as  pd
from pprint import pprint# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
#from gensim.models import CoherenceModel# spaCy for preprocessing
import spacy# Plotting tools
#import pyLDAvis
#import pyLDAvis.gensim
import matplotlib.pyplot as plt
%matplotlib inline


In [20]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
!pip3 install spacy

!python3 -m spacy download en #Language model

#pip3 install gensim # For topic modeling

#pip3 install pyLDAvis # For visualizing topic models

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
2022-10-09 12:53:49.088193: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 12.8 MB 2.0 MB/s 
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
# NLTK Stop words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

In [ ]:
def sent_to_words(sentences):
  for sentence in sentences:
    yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))            #deacc=True removes punctuations

In [ ]:
data_words = list(sent_to_words(data))
print(data_words[:1])

In [ ]:
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)
print(trigram_mod[bigram_mod[data_words[0]]])


In [ ]:
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [ ]:
data_words_nostops = remove_stopwords(data_words)
data_words_bigrams = make_bigrams(data_words_nostops)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [ ]:
print(data_lemmatized[:5])

[['believe'], ['record', 'new', 'covid', 'case'], ['county', 'long', 'require', 'covid', 'vaccine', 'employee'], ['capital', 'give', 'dose', 'covid', 'vaccine', 'people'], []]


In [ ]:
id2word = corpora.Dictionary(data_lemmatized)  
texts = data_lemmatized  
corpus = [id2word.doc2bow(text) for text in texts]  
print(corpus[:1])

[[(0, 1)]]


In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20, 
                                           random_state=100,
                                           update_every=1,
                                           chunksize=50,
                                           passes=50,
                                           alpha='auto',
                                           per_word_topics=True)

In [ ]:
from pprint import pprint
pprint(lda_model.print_topics())

[(0,
  '0.067*"die" + 0.051*"take" + 0.040*"want" + 0.037*"mask" + 0.026*"global" + '
  '0.025*"cause" + 0.024*"let" + 0.017*"comment" + 0.016*"issue" + '
  '0.013*"measure"'),
 (1,
  '0.055*"amp" + 0.046*"still" + 0.035*"good" + 0.029*"child" + 0.026*"week" + '
  '0.023*"month" + 0.022*"available" + 0.021*"help" + 0.020*"connect" + '
  '0.017*"repmtg"'),
 (2,
  '0.068*"work" + 0.056*"make" + 0.045*"stop" + 0.028*"read" + 0.027*"last" + '
  '0.024*"keep" + 0.023*"hard" + 0.021*"testing" + 0.021*"consider" + '
  '0.015*"enough"'),
 (3,
  '0.179*"vaccine" + 0.038*"patient" + 0.034*"long" + 0.033*"news" + '
  '0.024*"infect" + 0.019*"omicron" + 0.018*"resident" + 0.017*"warning" + '
  '0.013*"stimulus_check" + 0.013*"isolation"'),
 (4,
  '0.051*"study" + 0.042*"day" + 0.034*"lie" + 0.029*"see" + 0.020*"life" + '
  '0.020*"never" + 0.018*"lose" + 0.017*"follow" + 0.016*"home" + '
  '0.015*"look"'),
 (5,
  '0.066*"time" + 0.035*"think" + 0.032*"datum" + 0.027*"thank" + '
  '0.024*"source" +

In [ ]:
doc_lda = lda_model[corpus]

In [ ]:
# Compute Perplexity
print('\nPerplexity: ', lda_model.log_perplexity(corpus))  

In [ ]:
# Compute Coherence Score
from gensim.models import CoherenceModel# spaCy for preprocessing
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Coherence Score:  0.5747205099331698


In [ ]:
for idx, topic in lda_model.show_topics(formatted=False, num_topics=20,num_words= 100):
    print('Topic: {} \nWords: {}'.format(idx, '|'.join([w[0] for w in topic])))


Topic: 0 
Words: die|take|want|mask|global|cause|let|comment|issue|measure|avoid|thread|app|loss|stupid|reinstate|hide|proof|mind|past|hurt|wear|arrest|like|physically|experience|brother|deliver|restaurant|teacher|economic|growth|pand|whole|page|campaign|sudden|star|message|region|blood|scare|ad|reflect|slip|murder|appropriate|shut|panel|seriously|play|car|cancer|anywhere|cost|immigrant|football|radio|game|jail|nose|exacerbate|beg|donald_trump|everywhere|morning|alone|spike|cross|insanity|plant|taste|metric|living|injection|internet|opinion|survivor|framework|smell|alternative|bubble|regional|m|smear|argue|drjbhattacharya|horn|pain|viral|jab|profile|personnel|impairment|assault|market_size|advertising|barrier|hunger|clot
Topic: 1 
Words: amp|still|good|child|week|month|available|help|connect|repmtg|upon_time|safe|effective|reduction|prove|biden_ukriane|receive|doctor|family|serena_trump|ukriane|maga_biden|turn|check|booster_shot|boost|bedtime_story|filter|advise|door|flu|pretty|medicin

In [ ]:
from gensim.parsing.preprocessing import preprocess_string, strip_punctuation, strip_numeric

lda_topics = lda_model.show_topics(num_topics=20, num_words=50)

topics = []
filters = [lambda x: x.lower(), strip_punctuation, strip_numeric]

for topic in lda_topics:
    print(topic)
    topics.append(preprocess_string(topic[1], filters))

print(topics)

(0, '0.067*"die" + 0.051*"take" + 0.040*"want" + 0.037*"mask" + 0.026*"global" + 0.025*"cause" + 0.024*"let" + 0.017*"comment" + 0.016*"issue" + 0.013*"measure" + 0.013*"avoid" + 0.012*"thread" + 0.012*"app" + 0.012*"loss" + 0.009*"stupid" + 0.008*"reinstate" + 0.008*"hide" + 0.008*"proof" + 0.008*"mind" + 0.007*"past" + 0.007*"hurt" + 0.007*"wear" + 0.007*"arrest" + 0.007*"like" + 0.007*"physically" + 0.006*"experience" + 0.006*"brother" + 0.006*"deliver" + 0.006*"restaurant" + 0.005*"teacher" + 0.005*"economic" + 0.005*"growth" + 0.005*"pand" + 0.005*"whole" + 0.005*"page" + 0.005*"campaign" + 0.005*"sudden" + 0.005*"star" + 0.005*"message" + 0.005*"region" + 0.004*"blood" + 0.004*"scare" + 0.004*"ad" + 0.004*"reflect" + 0.004*"slip" + 0.004*"murder" + 0.004*"appropriate" + 0.004*"shut" + 0.004*"panel" + 0.004*"seriously"')
(1, '0.055*"amp" + 0.046*"still" + 0.035*"good" + 0.029*"child" + 0.026*"week" + 0.023*"month" + 0.022*"available" + 0.021*"help" + 0.020*"connect" + 0.017*"repmt

In [ ]:
!pip3 install pyLDAvis==2.1.2

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
# Visualize the topics
import pyLDAvis.gensim
#import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

/usr/local/lib/python3.7/dist-packages/pyLDAvis/_prepare.py:232: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  head(R).drop('saliency', 1)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
7     -0.380549  0.175858       1        1  11.051640
4     -0.160650 -0.369371       2        1   9.123652
1      0.019338  0.011984       3        1   6.187178
9     -0.004251  0.002225       4        1   6.030839
12    -0.004415 -0.007563       5        1   5.808395
3      0.016069  0.012568       6        1   4.752202
13     0.026641  0.010399       7        1   4.711831
5      0.031508  0.011646       8        1   4.651936
2      0.026067  0.011668       9        1   4.565354
14     0.011558  0.003148      10        1   4.548007
0      0.031601  0.014621      11        1   4.507710
18     0.039643  0.017722      12        1   4.436415
15     0.027102  0.007369      13        1   4.404828
16     0.040638  0.013311      14        1   4.186253
8      0.053138  0.017950      15        1   3.815061
11     0.028314  0.001875      16        1   3.723855
19     0.047261  0.014252      17        1   3.570896
6      0.042889  0.014844      18        1   3.527378
17     0.052218  0.017702      19        1   3.263361
10     0.055880  0.017793      20        1   3.133208, topic_info=               Term         Freq        Total Category  logprob  loglift
2             covid  5408.000000  5408.000000  Default  30.0000  30.0000
1              case   912.000000   912.000000  Default  29.0000  29.0000
36              get   816.000000   816.000000  Default  28.0000  28.0000
3               new   836.000000   836.000000  Default  27.0000  27.0000
9           vaccine   778.000000   778.000000  Default  26.0000  26.0000
...             ...          ...          ...      ...      ...      ...
1029           cold    10.578487    11.415709  Topic20  -5.6006   3.3869
4953  communication    10.246721    11.083953  Topic20  -5.6325   3.3846
1288   intelligence    10.232953    11.070176  Topic20  -5.6338   3.3845
9174         cbcnew    18.732808    20.542843  Topic20  -5.0292   3.3709
7181        faculty    11.497258    12.483000  Topic20  -5.5173   3.3809

[641 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
2305     16  0.984661       accord
4894      4  0.994534  accountable
84       18  0.994345       active
2455      7  0.978381       actual
1394     10  0.990024     actually
...     ...       ...          ...
69        5  0.997599         year
1456      9  0.970704    yesterday
534      14  0.974460          yet
41       19  0.968714           yo
2097     10  0.957461        young

[654 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[8, 5, 2, 10, 13, 4, 14, 6, 3, 15, 1, 19, 16, 17, 9, 12, 20, 7, 18, 11])

In [ ]:
pyLDAvis.save_html(vis, '2022_lda.html')

In [ ]:
!unzip "/content/mallet-2.0.8.zip"

In [ ]:
mallet_path = '/content/mallet-2.0.8/bin/mallet' # update this path
#ldamallet = gensim.models.wrappers.LdaMallet(mallet_path, 
#                                             corpus=corpus, 
#                                             num_topics=20,
#                                             id2word=id2word)

ldamallet = gensim.models.wrappers.LdaMallet(mallet_path,
                                           corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20,
                                           iterations=500
                                           )

/usr/local/lib/python3.7/dist-packages/smart_open/smart_open_lib.py:494: DeprecationWarning: This function is deprecated.  See https://github.com/RaRe-Technologies/smart_open/blob/develop/MIGRATING_FROM_OLDER_VERSIONS.rst for more information
  warnings.warn(message, category=DeprecationWarning)
/usr/local/lib/python3.7/dist-packages/smart_open/smart_open_lib.py:494: DeprecationWarning: This function is deprecated.  See https://github.com/RaRe-Technologies/smart_open/blob/develop/MIGRATING_FROM_OLDER_VERSIONS.rst for more information
  warnings.warn(message, category=DeprecationWarning)


In [ ]:
pprint(ldamallet.show_topics(num_topics=20,num_words=100,formatted=False))

# Compute Coherence Score
coherence_model_ldamallet = CoherenceModel(model=ldamallet, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_ldamallet = coherence_model_ldamallet.get_coherence()

print('\n Coherence Score: ', round(coherence_ldamallet, 2))

[(0,
  [('covid', 0.05407421219466716),
   ('pandemic', 0.022002610479209397),
   ('impact', 0.012120082043632295),
   ('global', 0.01006899123624837),
   ('health', 0.00857728883087824),
   ('economic', 0.008017900428864442),
   ('market', 0.007644974827521909),
   ('world', 0.007458512026850644),
   ('lockdown', 0.006712660824165579),
   ('report', 0.0063397352228230465),
   ('social', 0.0063397352228230465),
   ('crisis', 0.006153272422151781),
   ('job', 0.005966809621480515),
   ('government', 0.005593884020137983),
   ('due', 0.005407421219466716),
   ('industry', 0.005407421219466716),
   ('state', 0.005407421219466716),
   ('analysis', 0.00522095841879545),
   ('lose', 0.00522095841879545),
   ('end', 0.004661570016781652),
   ('share', 0.004661570016781652),
   ('wave', 0.004661570016781652),
   ('country', 0.004475107216110386),
   ('support', 0.004475107216110386),
   ('low', 0.004475107216110386),
   ('amp', 0.00428864441543912),
   ('growth', 0.004102181614767854),
   ('ex

In [ ]:
mallet_lda_model = gensim.models.wrappers.ldamallet.malletmodel2ldamodel(ldamallet)

In [ ]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(mallet_lda_model, corpus, id2word,sort_topics=False)

/usr/local/lib/python3.7/dist-packages/pyLDAvis/_prepare.py:232: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  head(R).drop('saliency', 1)


In [ ]:
print(vis.topic_order)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [ ]:
pyLDAvis.save_html(vis, '2022_lda_mallet.html')

In [ ]:
vis

PreparedData(topic_coordinates=              x         y  topics  cluster      Freq
topic                                               
0      0.000139  0.000605       1        1  4.957729
1     -0.000130  0.000024       2        1  5.029265
2      0.000219 -0.000093       3        1  5.006057
3     -0.000439 -0.000744       4        1  4.991864
4     -0.000019 -0.000260       5        1  5.039537
5     -0.000525 -0.000214       6        1  4.985984
6      0.000105  0.000091       7        1  4.987267
7     -0.000263  0.000002       8        1  4.981474
8     -0.000210 -0.000726       9        1  5.017666
9     -0.000185  0.000236      10        1  5.008625
10     0.000787 -0.000037      11        1  5.031838
11    -0.000198  0.000487      12        1  5.008244
12     0.000004 -0.000498      13        1  4.962519
13     0.000310  0.000047      14        1  4.996444
14     0.000317 -0.000125      15        1  5.016568
15     0.000506  0.000278      16        1  4.985773
16     0.000282  0.000312      17        1  5.026882
17     0.000497 -0.000298      18        1  5.008894
18    -0.000589  0.000361      19        1  4.961389
19    -0.000609  0.000554      20        1  4.995980, topic_info=                Term      Freq     Total Category  logprob  loglift
8437       infuriate  7.000000  7.000000  Default  30.0000  30.0000
4445         propose  7.000000  7.000000  Default  29.0000  29.0000
6906        sentinel  8.000000  8.000000  Default  28.0000  28.0000
8311   dontbeasucker  7.000000  7.000000  Default  27.0000  27.0000
4564         anxiety  8.000000  8.000000  Default  26.0000  26.0000
...              ...       ...       ...      ...      ...      ...
3479          mestre  0.507382  8.127811  Topic20  -9.1045   0.2228
6921   thanniversary  0.498317  7.896691  Topic20  -9.1225   0.2336
11696        vvriter  0.504027  8.164776  Topic20  -9.1111   0.2116
5674       traumatic  0.501536  8.099455  Topic20  -9.1161   0.2147
7437     patriotsong  0.499643  8.169589  Topic20  -9.1199   0.2023

[830 rows x 6 columns], token_table=       Topic      Freq            Term
term                                  
874        5  0.123445         abysmal
3481      20  0.128920        academia
6024      12  0.125761  accesstooxygen
2222      11  0.127246  acetylcysteine
6891      12  0.127644             ach
...      ...       ...             ...
2291       4  0.125313           worth
8599       3  0.130389             wth
8427       7  0.129107       wwecastle
6663      12  0.129735            xqdp
10643      6  0.126304            yard

[457 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])